# 🏦 Industry-Specific Retail Banking LLM Chatbot — Improved Capstone
**Fine-tuning FLAN-T5 on the Bitext Retail Banking dataset · Author: Harsh Parmar**

This notebook is a **rebuilt version** of the original submission, restructured to address the reviewer feedback:

> *"Perform thorough data analysis and text preprocessing. Do proper LLM hyperparameter tuning and implement LLM evaluation metrics to assess the model's performance. Demonstrate your working prototype in the explanation video with an interactive Gradio application."*

### How each feedback point is addressed

| Feedback | What was done | Section |
|---|---|---|
| Thorough **data analysis** | 15+ visualisations: category/intent balance, length distributions (words **and** T5 tokens), n-grams, word clouds, Zipf/vocabulary, response-style and noise analysis, duplicate/conflict checks | **2** |
| Proper **text preprocessing** | Unicode/whitespace/punctuation normalisation, placeholder normalisation, short/duplicate/outlier filtering with a step-by-step audit trail, leakage-safe **stratified train/val/test split**, data-driven max token lengths, task prefix | **3** |
| **LLM evaluation metrics** | ROUGE-1/2/L, BLEU, METEOR, BERTScore, perplexity, distinct-n, repetition/looping rate, length ratio, placeholder recall — plus a zero-shot baseline for comparison | **4, 5, 9, 10** |
| **LLM hyperparameter tuning** | Optuna search over learning-rate, batch size, weight decay, warm-up, scheduler **and** a decoding-parameter search (beams, repetition control, sampling) | **6, 8** |
| **Interactive Gradio** prototype | Chat tab with adjustable generation settings, a live *"Evaluate a response"* metrics tab, and a model-report tab | **11** |

### What was wrong in the original submission (and is fixed here)
| Original | Problem | Fix in this notebook |
|---|---|---|
| No EDA | Nothing known about the data before training | Section 2 |
| `max_length=128` for answers | Long answers were silently **truncated** during training | Lengths chosen from the data (Section 3.7) |
| Random 90/10 split, test set also used as validation | No clean held-out test set; paraphrase duplicates can leak | Stratified 80/10/10 split + leakage check |
| Only loss reported | Loss alone doesn't show answer quality | Full metric suite (Section 4) |
| Greedy decoding | The original output **looped** ("…such as your card number, account number … such as your card number,") | Repetition metrics + decoding search (Section 8) |
| Default hyper-parameters | Never tuned | Optuna study (Section 6) |
| Basic Gradio box | Single textbox only | Multi-tab Gradio app (Section 11) |

### ▶️ How to run
1. **Runtime → Change runtime type → T4 GPU**.
2. Choose `PROFILE` in the config cell: **`"quick"`** (default, a demo-friendly run) or **`"full"`** (more tuning trials, full data, more epochs — best results, much longer).
3. **Runtime → Run all.** Every chart and table is produced by the cells below, so all graphs and outputs are visible in Colab.
4. The last Gradio cell prints a public link you can use in your demo video.

## 0. Setup

In [ ]:
# Install extra libraries (Colab already has torch / transformers / datasets / gradio)
!pip install -q rouge_score sacrebleu bert_score nltk wordcloud optuna sentencepiece accelerate

In [ ]:
import os, re, gc, json, math, time, random, string, unicodedata, warnings
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 13, "axes.titleweight": "bold"})
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 200)

### ⚙️ Configuration
`PROFILE = "quick"` keeps total run-time reasonable on a free T4 GPU (tuning on a data subset, shorter final training).
Switch to `"full"` for the strongest model (more trials, all data, more epochs).

In [ ]:
SEED = 42
BASE_MODEL = "google/flan-t5-small"
HF_DATASET = "bitext/Bitext-retail-banking-llm-chatbot-training-dataset"
DATA_PATH = "/content/drive/My Drive/AlmaBetter/Specialization_Track/Deep Learning for NLP/bitext-retail-banking-llm-chatbot-training-dataset.csv"
PREFIX = "Answer the retail banking customer query: "   # task prefix (T5 is a text-to-text model)

PROFILE = "quick"   # "quick" or "full"
PROFILES = {
    "quick": dict(tune_train_n=3000, tune_val_n=400, n_trials=6,  tune_epochs=1,
                  final_train_n=10000, final_epochs=3, decode_val_n=150, test_eval_n=500),
    "full":  dict(tune_train_n=8000, tune_val_n=800, n_trials=12, tune_epochs=1,
                  final_train_n=None,  final_epochs=5, decode_val_n=300, test_eval_n=None),
}
CFG = PROFILES[PROFILE]
random.seed(SEED); np.random.seed(SEED)
print("Profile:", PROFILE, "->", CFG)

In [ ]:
import torch
from datasets import Dataset, load_dataset
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback, set_seed)
from tqdm.auto import tqdm
import optuna

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_BF16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
set_seed(SEED)
print("Device:", device, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none", "| bf16:", USE_BF16)
if device.type == "cpu":
    print("⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU, otherwise training will be extremely slow.")

In [ ]:
# Where to save outputs (model, metrics, processed data). Uses Google Drive when available.
OUTPUT_DIR = "/content/banking_llm_outputs"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = "/content/drive/MyDrive/banking_llm_outputs"
except Exception as e:
    print("Drive not mounted, saving locally:", e)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Outputs will be saved to:", OUTPUT_DIR)

## 1. Data Loading & First Look
The loader first tries your CSV on Google Drive (same path as the original submission). If the file is not found it
downloads the identical **Bitext Retail Banking** dataset from the Hugging Face Hub, so the notebook runs anywhere.

In [ ]:
def load_raw():
    if os.path.exists(DATA_PATH):
        print("Loading CSV from Google Drive:", DATA_PATH)
        return pd.read_csv(DATA_PATH)
    print("CSV not found on Drive -> downloading from Hugging Face Hub:", HF_DATASET)
    return load_dataset(HF_DATASET, split="train").to_pandas()

raw_df = load_raw()
print("Loaded:", raw_df.shape)

In [ ]:
REQUIRED = ["instruction", "response"]
missing_cols = [c for c in REQUIRED if c not in raw_df.columns]
assert not missing_cols, f"Dataset must contain {REQUIRED}; missing {missing_cols}"

has_flags = "flags" in raw_df.columns
for c in ["category", "intent"]:
    if c not in raw_df.columns:
        raw_df[c] = "unknown"
        print(f"Column '{c}' not found -> filled with 'unknown'")
raw_df = raw_df[[c for c in ["flags", "instruction", "category", "intent", "response"] if c in raw_df.columns]].copy()

print(f"Rows: {len(raw_df):,} | Columns: {list(raw_df.columns)}")
display(raw_df.head())

In [ ]:
raw_df.info()
display(raw_df.describe(include="object").T)

## 2. Exploratory Data Analysis (EDA)
Each chart below is followed by a short note on **why this chart** and **what to look for**; the computed findings are
printed from the data itself so they always match what you see.

### 2.1 Data quality audit

In [ ]:
txt_cols = ["instruction", "response"]
quality = pd.DataFrame({
    "missing":      raw_df[txt_cols].isna().sum(),
    "empty_string": [(raw_df[c].fillna("").str.strip() == "").sum() for c in txt_cols],
    "unique":       [raw_df[c].nunique() for c in txt_cols],
}, index=txt_cols)
display(quality)

exact_dup = raw_df.duplicated(subset=txt_cols).sum()
instr_dup = raw_df.duplicated(subset=["instruction"]).sum()
conflict = (raw_df.groupby("instruction")["response"].nunique() > 1).sum()
print(f"Exact duplicate (instruction, response) pairs : {exact_dup:,}")
print(f"Duplicate instructions (same question text)    : {instr_dup:,}")
print(f"Instructions mapped to >1 different responses  : {conflict:,}")

**Why:** dirty data (missing/duplicate/conflicting pairs) silently hurts fine-tuning and **leaks between train and test** when the same
question appears in both. These counts decide what the preprocessing in Section 3 must remove.

### 2.2 Category & intent balance

In [ ]:
cat_counts = raw_df["category"].value_counts()
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.barplot(x=cat_counts.values, y=cat_counts.index, hue=cat_counts.index, palette="viridis", legend=False, ax=ax)
for c in ax.containers:
    ax.bar_label(c, padding=3)
ax.set_title("Samples per Category"); ax.set_xlabel("Number of samples"); ax.set_ylabel("")
plt.tight_layout(); plt.show()

In [ ]:
int_counts = raw_df["intent"].value_counts()
top_int = int_counts.head(25)
fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(x=top_int.values, y=top_int.index, hue=top_int.index, palette="mako", legend=False, ax=ax)
ax.set_title(f"Top {len(top_int)} Intents (of {len(int_counts)} total)"); ax.set_xlabel("Number of samples"); ax.set_ylabel("")
plt.tight_layout(); plt.show()
print(f"{len(int_counts)} intents | largest/smallest ratio = {int_counts.max()/max(1,int_counts.min()):.1f}x | median = {int_counts.median():.0f} samples/intent")

In [ ]:
if 1 < raw_df["intent"].nunique() <= 40 and raw_df["category"].nunique() > 1:
    ct = pd.crosstab(raw_df["intent"], raw_df["category"])
    plt.figure(figsize=(11, 9))
    sns.heatmap(ct, cmap="YlGnBu", annot=ct.shape[0] * ct.shape[1] <= 200, fmt="d", cbar_kws={"label": "samples"})
    plt.title("Intent × Category (sample counts)"); plt.tight_layout(); plt.show()
else:
    print("Intent × category heatmap skipped (too many intents or single category).")

**Why:** bar charts show class balance; the heatmap shows which intents belong to which banking category.
**Look for:** heavy imbalance (the model will be better on frequent intents) — this is why Section 3 uses a **stratified split by intent**
so every intent appears in train, validation and test.

### 2.3 Text length analysis (words)

In [ ]:
def n_words(s):
    return len(re.findall(r"\S+", s)) if isinstance(s, str) else 0

raw_df["instr_words"] = raw_df["instruction"].map(n_words)
raw_df["resp_words"] = raw_df["response"].map(n_words)
raw_df["instr_chars"] = raw_df["instruction"].fillna("").str.len()
raw_df["resp_chars"] = raw_df["response"].fillna("").str.len()

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
sns.histplot(raw_df["instr_words"], bins=40, kde=True, color="#4C72B0", ax=axes[0])
axes[0].set_title("Customer Query Length (words)"); axes[0].set_xlabel("words")
sns.histplot(raw_df["resp_words"], bins=40, kde=True, color="#DD8452", ax=axes[1])
axes[1].set_title("Bot Response Length (words)"); axes[1].set_xlabel("words")
plt.tight_layout(); plt.show()
display(raw_df[["instr_words", "resp_words", "instr_chars", "resp_chars"]].describe().round(1).T)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
order = raw_df.groupby("category")["resp_words"].median().sort_values(ascending=False).index
sns.boxplot(data=raw_df, x="resp_words", y="category", order=order, orient="h", showfliers=False, palette="Set2", hue="category", legend=False, ax=axes[0])
axes[0].set_title("Response Length by Category"); axes[0].set_xlabel("words"); axes[0].set_ylabel("")
sns.boxplot(data=raw_df, x="instr_words", y="category", order=order, orient="h", showfliers=False, palette="Set3", hue="category", legend=False, ax=axes[1])
axes[1].set_title("Query Length by Category"); axes[1].set_xlabel("words"); axes[1].set_ylabel("")
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(7, 5.5))
hb = plt.hexbin(raw_df["instr_words"], raw_df["resp_words"], gridsize=40, cmap="magma", mincnt=1)
plt.colorbar(hb, label="count")
plt.xlabel("Query length (words)"); plt.ylabel("Response length (words)")
plt.title("Query length vs Response length")
plt.tight_layout(); plt.show()
rho = raw_df[["instr_words", "resp_words"]].corr(method="spearman").iloc[0, 1]
print(f"Spearman correlation (query length vs response length): {rho:.3f}")

**Why:** histograms show the shape; boxplots compare categories; the hexbin checks whether longer questions get longer answers.
**Look for:** answers are far longer than questions, which means the **decoder** (not the encoder) dictates the sequence-length budget —
exactly why the original `max_length=128` for answers is risky.

### 2.4 Token-length analysis with the T5 tokenizer
Models see *tokens*, not words. We measure real T5 token lengths to choose `max_length` from data instead of guessing.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
print("Tokenizer loaded:", tokenizer.__class__.__name__, "| vocab size:", tokenizer.vocab_size)

In [ ]:
def token_lengths(texts, bs=2000):
    texts = list(texts); out = []
    for i in range(0, len(texts), bs):
        ids = tokenizer(texts[i:i + bs], add_special_tokens=True)["input_ids"]
        out.extend(len(x) for x in ids)
    return np.array(out)

src_len = token_lengths(raw_df["instruction"].fillna(""))
tgt_len = token_lengths(raw_df["response"].fillna(""))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
for ax, arr, name, col in [(axes[0], src_len, "Query", "#4C72B0"), (axes[1], tgt_len, "Response", "#DD8452")]:
    sns.histplot(arr, bins=50, color=col, ax=ax)
    ax.axvline(128, color="red", ls="--", lw=2, label="128 (original max_length)")
    ax.axvline(np.percentile(arr, 95), color="green", ls=":", lw=2, label=f"95th pct = {np.percentile(arr, 95):.0f}")
    ax.axvline(np.percentile(arr, 99), color="purple", ls=":", lw=2, label=f"99th pct = {np.percentile(arr, 99):.0f}")
    ax.set_title(f"{name} Token Length (T5 tokenizer)"); ax.set_xlabel("tokens"); ax.legend()
plt.tight_layout(); plt.show()

print(f"Queries   > 128 tokens: {(src_len > 128).mean() * 100:.2f}%")
print(f"Responses > 128 tokens: {(tgt_len > 128).mean() * 100:.2f}%   <-- these answers were TRUNCATED in the original training")
print(f"Max query tokens: {src_len.max()} | max response tokens: {tgt_len.max()}")

**Insight:** any response beyond the red line was cut off during the original training, so the model never learned how long answers should *end* —
a plausible contributor to the rambling/looping output seen before. Section 3 sets the limits from the green/purple percentile lines instead.

### 2.5 Most frequent words and phrases (n-grams)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def top_ngrams(texts, ngram=(1, 1), k=20, sample=20000):
    s = pd.Series(list(texts)).dropna()
    s = s.sample(min(sample, len(s)), random_state=SEED)
    vec = CountVectorizer(stop_words="english", ngram_range=ngram, min_df=2)
    X = vec.fit_transform(s)
    freq = np.asarray(X.sum(axis=0)).ravel()
    vocab = np.array(vec.get_feature_names_out())
    idx = freq.argsort()[::-1][:k]
    return pd.DataFrame({"ngram": vocab[idx], "count": freq[idx]})

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
specs = [("instruction", (1, 1), "Top words in QUERIES", "Blues_r", axes[0, 0]),
         ("instruction", (2, 2), "Top bigrams in QUERIES", "Blues_r", axes[0, 1]),
         ("response", (1, 1), "Top words in RESPONSES", "Oranges_r", axes[1, 0]),
         ("response", (2, 2), "Top bigrams in RESPONSES", "Oranges_r", axes[1, 1])]
for col, ng, title, pal, ax in specs:
    d = top_ngrams(raw_df[col], ng)
    sns.barplot(data=d, x="count", y="ngram", hue="ngram", palette=pal, legend=False, ax=ax)
    ax.set_title(title); ax.set_ylabel("")
plt.tight_layout(); plt.show()

In [ ]:
from wordcloud import WordCloud, STOPWORDS
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, col, title in [(axes[0], "instruction", "Word Cloud — Customer Queries"), (axes[1], "response", "Word Cloud — Bot Responses")]:
    col_series = raw_df[col].dropna()
    text = " ".join(col_series.sample(min(5000, len(col_series)), random_state=SEED))
    wc = WordCloud(width=900, height=500, background_color="white", stopwords=STOPWORDS, collocations=False, colormap="viridis").generate(text)
    ax.imshow(wc, interpolation="bilinear"); ax.axis("off"); ax.set_title(title)
plt.tight_layout(); plt.show()

**Why:** n-gram bars quantify vocabulary; word clouds give a quick visual of the domain (cards, accounts, loans, …).
**Look for:** repeated boiler-plate phrases in responses ("customer support", "I'm here to…") — the model will learn them strongly, and they are also where repetition tends to creep in.

### 2.6 Vocabulary richness & Zipf's law

In [ ]:
def word_tokens(series, n=20000):
    s = series.dropna().sample(min(n, len(series.dropna())), random_state=SEED)
    return re.findall(r"[a-z']+", " ".join(s).lower())

vocab_stats = {}
fig, ax = plt.subplots(figsize=(8, 5))
for name, col, color in [("Queries", "instruction", "#4C72B0"), ("Responses", "response", "#DD8452")]:
    toks = word_tokens(raw_df[col]); cnt = Counter(toks)
    freqs = np.array(sorted(cnt.values(), reverse=True))
    ax.loglog(np.arange(1, len(freqs) + 1), freqs, label=name, color=color, lw=2)
    vocab_stats[name] = {"tokens": len(toks), "vocab size": len(cnt),
                         "type-token ratio": round(len(cnt) / len(toks), 4),
                         "hapax (appear once) %": round(100 * sum(1 for v in cnt.values() if v == 1) / len(cnt), 1)}
ax.set_xlabel("word rank (log)"); ax.set_ylabel("frequency (log)"); ax.set_title("Zipf plot — rank vs frequency"); ax.legend()
plt.tight_layout(); plt.show()
display(pd.DataFrame(vocab_stats).T)

**Why:** a log-log rank/frequency curve is the standard check that text behaves like natural language; the table shows vocabulary size and how many words are rare.
**Look for:** queries have a *larger share of rare words* (typos, slang), which is what makes real-world customer input hard — a robustness issue examined again in Section 10.

### 2.7 Response style analysis

In [ ]:
resp = raw_df["response"].fillna("")
openings = resp.str.split().str[:4].str.join(" ").value_counts().head(12)

features = {
    "Contains numbered steps":    resp.str.contains(r"\b1\.\s", regex=True),
    "Contains {{placeholder}}":   resp.str.contains(r"\{\{", regex=True),
    "Apologises (sorry/apolog)":  resp.str.contains(r"sorry|apolog", case=False, regex=True),
    "Mentions customer support":  resp.str.contains(r"customer support|contact", case=False, regex=True),
    "Contains a question mark":   resp.str.contains(r"\?", regex=True),
    "Contains exclamation mark":  resp.str.contains(r"!", regex=True),
}
rates = pd.Series({k: v.mean() * 100 for k, v in features.items()}).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(x=openings.values, y=openings.index, hue=openings.index, palette="rocket", legend=False, ax=axes[0])
axes[0].set_title("12 Most Common Response Openings (first 4 words)"); axes[0].set_xlabel("count"); axes[0].set_ylabel("")
sns.barplot(x=rates.values, y=rates.index, hue=rates.index, palette="crest", legend=False, ax=axes[1])
for c in axes[1].containers: axes[1].bar_label(c, fmt="%.1f%%", padding=3)
axes[1].set_title("Response Style Features (% of responses)"); axes[1].set_xlabel("% of responses"); axes[1].set_ylabel("")
plt.tight_layout(); plt.show()

**Why:** knowing how responses are *written* (templated openings, numbered steps, placeholders) tells us what the model will imitate and what a good metric should check.
`{{placeholder}}` tokens (e.g. account-number fields) become a **domain-specific evaluation metric** in Section 4.

### 2.8 Query noise analysis

In [ ]:
q = raw_df["instruction"].fillna("")
noise = pd.Series({
    "No ending punctuation": ~q.str.contains(r"[.?!]\s*$", regex=True),
    "All lower-case":        q.str.islower(),
    "Contains digits":       q.str.contains(r"\d", regex=True),
    "Contains special chars":q.str.contains(r"[^\w\s.,?!'\-]", regex=True),
    "Has double spaces":     q.str.contains(r"\s{2,}", regex=True),
}).map(lambda s: s.mean() * 100).sort_values()

n_plots = 2 if has_flags else 1
fig, axes = plt.subplots(1, n_plots, figsize=(8 * n_plots, 4.5))
axes = np.atleast_1d(axes)
sns.barplot(x=noise.values, y=noise.index, hue=noise.index, palette="flare", legend=False, ax=axes[0])
for c in axes[0].containers: axes[0].bar_label(c, fmt="%.1f%%", padding=3)
axes[0].set_title("How 'messy' are customer queries?"); axes[0].set_xlabel("% of queries"); axes[0].set_ylabel("")
if has_flags:
    fl = Counter("".join(raw_df["flags"].dropna().astype(str))).most_common()
    fl_df = pd.DataFrame(fl, columns=["flag", "count"])
    sns.barplot(data=fl_df, x="flag", y="count", hue="flag", palette="Spectral", legend=False, ax=axes[1])
    axes[1].set_title("Linguistic-variation flags in the dataset (count per letter)")
plt.tight_layout(); plt.show()

**Why:** real customers type in lower-case, skip punctuation and make typos. The dataset's *flags* column marks these linguistic variations,
which lets us test **robustness per variation** in Section 10.

In [ ]:
print("=" * 70)
print("KEY EDA FINDINGS (computed from the data)")
print("=" * 70)
print(f"1. {len(raw_df):,} rows | {raw_df['category'].nunique()} categories | {raw_df['intent'].nunique()} intents "
      f"(largest/smallest intent = {int_counts.max()/max(1,int_counts.min()):.1f}x).")
print(f"2. Median query = {raw_df['instr_words'].median():.0f} words; median response = {raw_df['resp_words'].median():.0f} words "
      f"({raw_df['resp_words'].median()/max(1,raw_df['instr_words'].median()):.1f}x longer).")
print(f"3. {(tgt_len > 128).mean()*100:.1f}% of responses exceed 128 T5 tokens -> truncation risk in the original setup.")
print(f"4. Exact duplicate pairs: {exact_dup:,}; duplicate queries: {instr_dup:,}; queries with conflicting answers: {conflict:,}.")
print(f"5. {rates.get('Contains {{placeholder}}', 0):.1f}% of responses contain {{placeholders}}; "
      f"{rates.get('Apologises (sorry/apolog)', 0):.1f}% start/contain apologies -> highly templated style.")
print(f"6. {noise.get('All lower-case', 0):.1f}% of queries are all lower-case and {noise.get('No ending punctuation', 0):.1f}% lack final punctuation -> noisy real-world input.")
print("=> Implication: clean text carefully, split by intent (no leakage), pick max lengths from data, and evaluate with more than loss.")

## 3. Text Preprocessing & Dataset Preparation

### 3.1 Preprocessing decisions
| Step | Applied? | Reason |
|---|---|---|
| Unicode NFKC normalisation, remove zero-width chars, non-breaking spaces | ✅ | Invisible characters create different tokens for identical text |
| Collapse newlines/tabs/multiple spaces | ✅ | T5's SentencePiece has no newline token; extra spaces add noise |
| Collapse repeated punctuation (`!!!!` → `!`) | ✅ | Reduces rare tokens |
| Normalise `{{ Placeholder }}` spacing | ✅ | One canonical form so the model learns the pattern consistently |
| **Lower-casing** | ❌ | T5 is case-sensitive; answers must keep proper nouns/sentence case |
| **Spell-correcting queries** | ❌ | Real users make typos — the model should learn to cope with them |
| **Stop-word removal / stemming** | ❌ | Harmful for generative models; they need full sentences |
| Drop empty / very short rows | ✅ | No learning signal |
| Drop duplicate pairs **and duplicate queries** | ✅ | Prevents train/test leakage and conflicting targets |
| Drop extreme length outliers | ✅ | Keeps batches efficient without truncating real answers |
| Add task prefix | ✅ | T5-style instruction improves zero/few-shot behaviour |
| Stratified split by intent | ✅ | Every intent is represented in train/val/test |

### 3.2 Cleaning functions

In [ ]:
_ZERO_WIDTH = re.compile(r"[\u200b\u200c\u200d\u2060\ufeff]")

def clean_text(s, is_response=False):
    """Normalise a raw string. Case and typos are deliberately preserved."""
    if not isinstance(s, str):
        return ""
    s = unicodedata.normalize("NFKC", s)
    s = _ZERO_WIDTH.sub("", s)
    s = s.replace("\u00a0", " ")
    s = re.sub(r"[\r\n\t]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"([!?.,])\1{2,}", r"\1", s)            # '!!!!' -> '!'
    s = re.sub(r"\s+([,.!?;:])", r"\1", s)              # 'card !' -> 'card!'
    if is_response:
        s = re.sub(r"\{\{\s*([^{}]+?)\s*\}\}", lambda m: "{{" + m.group(1).strip() + "}}", s)
    return s

def norm_key(s):
    """Aggressive key used ONLY for duplicate detection (never fed to the model)."""
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", "", s.lower())).strip()

demo = pd.DataFrame({"before": [
    "  i  cant\u00a0find my   card!!!!  ",
    "WHERE is my\nstatement ???",
    "check   my balance .....please",
    "Please {{ Account Number }}  to continue\t now",
]})
demo["after"] = demo["before"].map(lambda s: clean_text(s, is_response=True))
display(demo.assign(before=demo["before"].map(repr), after=demo["after"].map(repr)))

### 3.3 Apply the pipeline with a full audit trail

In [ ]:
log = []
def step(name, d):
    log.append({"step": name, "rows_remaining": len(d)})
    return d

df = step("1. raw data", raw_df.copy())
df = step("2. drop missing text", df.dropna(subset=["instruction", "response"]))

df["instruction"] = df["instruction"].map(lambda s: clean_text(s))
df["response"] = df["response"].map(lambda s: clean_text(s, is_response=True))

df = step("3. drop empty after cleaning", df[(df["instruction"].str.len() > 0) & (df["response"].str.len() > 0)])
df = step("4. drop too short (query<2 words or answer<5 words)",
          df[(df["instruction"].str.split().str.len() >= 2) & (df["response"].str.split().str.len() >= 5)])

df["_ikey"] = df["instruction"].map(norm_key)
df = step("5. drop exact duplicate (query, answer) pairs", df.drop_duplicates(subset=["_ikey", "response"]))
df = step("6. one answer per unique query (prevents leakage & conflicts)", df.drop_duplicates(subset=["_ikey"]))

df["src_tok"] = token_lengths(df["instruction"])
df["tgt_tok"] = token_lengths(df["response"])
hi_t = int(np.percentile(df["tgt_tok"], 99.5)); hi_s = int(np.percentile(df["src_tok"], 99.9))
df = step(f"7. drop length outliers (answer>{hi_t} or query>{hi_s} tokens)", df[(df["tgt_tok"] <= hi_t) & (df["src_tok"] <= hi_s)])
df = df.reset_index(drop=True)

log_df = pd.DataFrame(log)
log_df["removed"] = (-log_df["rows_remaining"].diff()).fillna(0).astype(int)
display(log_df)

fig, ax = plt.subplots(figsize=(11, 4.5))
sns.barplot(data=log_df, y="step", x="rows_remaining", hue="step", palette="Blues_d", legend=False, ax=ax)
for c in ax.containers: ax.bar_label(c, padding=3)
lo = log_df["rows_remaining"].min(); hi = log_df["rows_remaining"].max()
ax.set_xlim(max(0, lo - (hi - lo) * 2 - 10), hi * 1.08)
ax.set_title("Preprocessing funnel — rows remaining after each step"); ax.set_ylabel(""); ax.set_xlabel("rows")
plt.tight_layout(); plt.show()
print(f"Kept {len(df):,} of {len(raw_df):,} rows ({len(df)/len(raw_df)*100:.1f}%).")

### 3.4 Before vs after

In [ ]:
df["instr_words"] = df["instruction"].map(n_words)
df["resp_words"] = df["response"].map(n_words)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for ax, col, title in [(axes[0], "instr_words", "Query length (words)"), (axes[1], "resp_words", "Response length (words)")]:
    sns.histplot(raw_df[col], bins=40, stat="density", color="grey", alpha=0.5, label="raw", ax=ax)
    sns.histplot(df[col], bins=40, stat="density", color="#2a9d8f", alpha=0.5, label="cleaned", ax=ax)
    ax.set_title(title); ax.legend()
plt.tight_layout(); plt.show()

ex = raw_df.sample(3, random_state=1).index
cmp = pd.DataFrame({"raw_instruction": raw_df.loc[ex, "instruction"].values,
                    "cleaned_instruction": raw_df.loc[ex, "instruction"].map(lambda s: clean_text(s)).values})
display(cmp)

### 3.5 Task prefix, stratified split & leakage check

In [ ]:
from sklearn.model_selection import train_test_split

df["source"] = PREFIX + df["instruction"]
df["target"] = df["response"]
df["src_tok"] = token_lengths(df["source"])      # now includes the prefix

def stratified_split(d, col, seed=SEED):
    """80/10/10 split stratified by `col`; intents with <10 rows go to train only."""
    if col is None or d[col].nunique() < 2:
        tr, tmp = train_test_split(d, test_size=0.2, random_state=seed)
        va, te = train_test_split(tmp, test_size=0.5, random_state=seed)
        return tr, va, te
    vc = d[col].value_counts()
    rare = d[d[col].map(vc) < 10]
    main = d.drop(rare.index)
    tr, tmp = train_test_split(main, test_size=0.2, stratify=main[col], random_state=seed)
    va, te = train_test_split(tmp, test_size=0.5, stratify=tmp[col], random_state=seed)
    return pd.concat([tr, rare]), va, te

train_df, val_df, test_df = stratified_split(df, "intent" if df["intent"].nunique() > 1 else None)
train_df, val_df, test_df = [d.reset_index(drop=True) for d in (train_df, val_df, test_df)]
print(f"Train: {len(train_df):,} | Validation: {len(val_df):,} | Test: {len(test_df):,}")

# Leakage check: no query may appear in more than one split
tr_k, va_k, te_k = set(train_df["_ikey"]), set(val_df["_ikey"]), set(test_df["_ikey"])
print(f"Overlap train∩test: {len(tr_k & te_k)} | train∩val: {len(tr_k & va_k)} | val∩test: {len(va_k & te_k)}  (must all be 0)")

# Does the split preserve the category distribution?
dist = pd.concat({n: d["category"].value_counts(normalize=True) * 100
                  for n, d in [("train", train_df), ("validation", val_df), ("test", test_df)]}, axis=1)
dist.plot(kind="bar", figsize=(11, 4.5), width=0.8, colormap="Set2")
plt.title("Category share (%) per split — stratification check"); plt.ylabel("% of split"); plt.xlabel("")
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

### 3.6 Choose max sequence lengths from the data (not by guessing)

In [ ]:
MAX_SRC = int(min(128, np.ceil(np.percentile(train_df["src_tok"], 99.9) / 8) * 8))
MAX_TGT = int(min(384, np.ceil(train_df["tgt_tok"].max() / 8) * 8))
print(f"MAX_SRC = {MAX_SRC} tokens | MAX_TGT = {MAX_TGT} tokens   (original submission used 128 for both)")
print(f"Answers truncated with MAX_TGT: {(train_df['tgt_tok'] > MAX_TGT).mean() * 100:.2f}%  | "
      f"with the old 128 limit: {(train_df['tgt_tok'] > 128).mean() * 100:.2f}%")

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    d.drop(columns=["_ikey"]).to_csv(os.path.join(OUTPUT_DIR, f"processed_{name}.csv"), index=False)
print("Processed splits saved to", OUTPUT_DIR)

### 3.7 Tokenisation (dynamic padding — much faster than padding everything to 128)

In [ ]:
def tok_fn(batch):
    enc = tokenizer(batch["source"], max_length=MAX_SRC, truncation=True)
    lab = tokenizer(text_target=batch["target"], max_length=MAX_TGT, truncation=True)
    enc["labels"] = lab["input_ids"]
    return enc

def to_tokenized(d):
    ds = Dataset.from_pandas(d[["source", "target"]].reset_index(drop=True))
    return ds.map(tok_fn, batched=True, remove_columns=ds.column_names)

train_tok, val_tok, test_tok = to_tokenized(train_df), to_tokenized(val_df), to_tokenized(test_df)
print(train_tok)

ex = train_tok[0]
print("\nInput tokens :", tokenizer.convert_ids_to_tokens(ex["input_ids"])[:25], "...")
print("Input length :", len(ex["input_ids"]), "| Label length:", len(ex["labels"]))
print("Decoded input:", tokenizer.decode(ex["input_ids"], skip_special_tokens=True))

## 4. LLM Evaluation Framework
Loss alone does not tell us if answers are *good*. We therefore evaluate generated answers against the reference answers with complementary metrics:

| Metric | What it measures | Good value |
|---|---|---|
| **Perplexity** | How surprised the model is by the reference answers (`exp(loss)`) | lower ↓ |
| **ROUGE-1 / 2 / L** | Word, bigram and longest-common-subsequence overlap (recall-oriented) | higher ↑ |
| **BLEU** | n-gram *precision* with brevity penalty | higher ↑ |
| **METEOR** | Overlap with stemming/synonym matching | higher ↑ |
| **BERTScore (F1)** | **Semantic** similarity using contextual embeddings — catches paraphrases that overlap metrics miss | higher ↑ |
| **Distinct-1 / Distinct-2** | Vocabulary diversity of generated text | higher ↑ |
| **Repetition rate / Looping rate** | Share of repeated trigrams inside an answer; share of answers that loop (>30% repeated) | lower ↓ |
| **Length ratio** | Mean generated length ÷ mean reference length | ≈ 1.0 |
| **Placeholder recall** | Share of reference `{{placeholders}}` reproduced (domain-specific) | higher ↑ |

> ⚠️ Reference-based metrics under-rate good answers that are worded differently from the reference. That is why BERTScore and a qualitative review (Section 10) are reported alongside ROUGE/BLEU.

In [ ]:
from rouge_score import rouge_scorer
import sacrebleu, nltk
from nltk.translate.meteor_score import meteor_score
for pkg in ["wordnet", "omw-1.4"]:
    try: nltk.download(pkg, quiet=True)
    except Exception as e: print("nltk download issue:", pkg, e)

_rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
PLACEHOLDER = re.compile(r"\{\{[^{}]+\}\}")

def trigram_repetition(text):
    w = text.lower().split()
    if len(w) < 4: return 0.0
    tri = list(zip(w, w[1:], w[2:]))
    return 1 - len(set(tri)) / len(tri)

def distinct_n(texts, n):
    grams = []
    for t in texts:
        w = t.lower().split()
        grams += list(zip(*[w[i:] for i in range(n)]))
    return len(set(grams)) / max(1, len(grams))

def placeholder_recall(preds, refs):
    scores = []
    for p, r in zip(preds, refs):
        ph = PLACEHOLDER.findall(r)
        if ph:
            scores.append(sum(1 for x in ph if x in p) / len(ph))
    return float(np.mean(scores)) if scores else float("nan")

def compute_metrics_suite(preds, refs, use_bert=True, bert_model="distilbert-base-uncased"):
    """Return (summary dict, per-example DataFrame)."""
    preds = [p if isinstance(p, str) else "" for p in preds]
    refs = list(refs)
    rows = []
    for p, r in zip(preds, refs):
        sc = _rouge.score(r, p)
        try:    met = meteor_score([r.split()], p.split())
        except Exception: met = np.nan
        rows.append(dict(rouge1=sc["rouge1"].fmeasure, rouge2=sc["rouge2"].fmeasure, rougeL=sc["rougeL"].fmeasure,
                         bleu=sacrebleu.sentence_bleu(p, [r]).score / 100, meteor=met,
                         pred_words=len(p.split()), ref_words=len(r.split()), repetition=trigram_repetition(p)))
    per = pd.DataFrame(rows)
    if use_bert:
        from bert_score import score as bert_score_fn
        _, _, F = bert_score_fn(preds, refs, model_type=bert_model, lang="en", batch_size=64,
                                device=str(globals().get("device", "cpu")), verbose=False)
        per["bertscore_f1"] = F.numpy()
    summary = {
        "ROUGE-1": per["rouge1"].mean(), "ROUGE-2": per["rouge2"].mean(), "ROUGE-L": per["rougeL"].mean(),
        "BLEU": sacrebleu.corpus_bleu(preds, [refs]).score / 100,
        "METEOR": per["meteor"].mean(),
        "BERTScore-F1": per["bertscore_f1"].mean() if use_bert else np.nan,
        "Distinct-1": distinct_n(preds, 1), "Distinct-2": distinct_n(preds, 2),
        "Repetition rate": per["repetition"].mean(), "Looping rate": (per["repetition"] > 0.3).mean(),
        "Length ratio": per["pred_words"].mean() / max(1e-9, per["ref_words"].mean()),
        "Placeholder recall": placeholder_recall(preds, refs),
    }
    return summary, per

# --- tiny self-test so you can see the metrics behave sensibly ---
_refs = ["To activate your card, log in to the mobile app and select Activate Card."]
_good = ["To activate your card, open the mobile app and choose Activate Card."]
_bad  = ["card card card number account number card card number account number"]
for name, p in [("good paraphrase", _good), ("degenerate / looping", _bad)]:
    s, _ = compute_metrics_suite(p, _refs, use_bert=False)
    print(f"{name:22s} ROUGE-L={s['ROUGE-L']:.3f}  BLEU={s['BLEU']:.3f}  METEOR={s['METEOR']:.3f}  repetition={s['Repetition rate']:.2f}")

In [ ]:
@torch.no_grad()
def generate_batch(model_, prompts, batch_size=32, max_new_tokens=None, **gen_kwargs):
    """Batched generation (sorted by length for speed, original order restored)."""
    model_.eval()
    max_new_tokens = max_new_tokens or MAX_TGT
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    outs = [None] * len(prompts)
    for s in tqdm(range(0, len(order), batch_size), desc="generating", leave=False):
        idx = order[s:s + batch_size]
        enc = tokenizer([prompts[i] for i in idx], return_tensors="pt", padding=True, truncation=True, max_length=MAX_SRC).to(device)
        out = model_.generate(**enc, max_new_tokens=max_new_tokens, **gen_kwargs)
        for i, t in zip(idx, tokenizer.batch_decode(out, skip_special_tokens=True)):
            outs[i] = t
    return outs

@torch.no_grad()
def compute_eval_loss(model_, ds, batch_size=32):
    """Token-weighted mean cross-entropy over a tokenised dataset (used for perplexity)."""
    model_.eval()
    coll = DataCollatorForSeq2Seq(tokenizer, model=model_, pad_to_multiple_of=8)
    total, n_tok = 0.0, 0
    for i in range(0, len(ds), batch_size):
        feats = [ds[j] for j in range(i, min(i + batch_size, len(ds)))]
        batch = {k: v.to(device) for k, v in coll(feats).items()}
        ntok = (batch["labels"] != -100).sum().item()
        total += model_(**batch).loss.item() * ntok
        n_tok += ntok
    return total / n_tok

GREEDY = dict(num_beams=1, do_sample=False)
RESULTS, PER_EXAMPLE = {}, {}          # collected for the final comparison
print("Evaluation helpers ready.")

## 5. Baseline: zero-shot FLAN-T5-small (before fine-tuning)
A fair evaluation needs a reference point. We run the **un-tuned** base model on the same test questions to show what fine-tuning actually adds.

In [ ]:
test_eval_df = test_df if CFG["test_eval_n"] is None else test_df.sample(n=min(CFG["test_eval_n"], len(test_df)), random_state=SEED)
test_eval_df = test_eval_df.reset_index(drop=True)
print(f"Evaluating on {len(test_eval_df):,} held-out test questions.")

base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL).to(device)
baseline_preds = generate_batch(base_model, list(test_eval_df["source"]), **GREEDY)
baseline_loss = compute_eval_loss(base_model, test_tok)
baseline_ppl = math.exp(baseline_loss)
print(f"Baseline test loss = {baseline_loss:.3f} | perplexity = {baseline_ppl:.2f}")

summ, per = compute_metrics_suite(baseline_preds, test_eval_df["target"])
summ["Perplexity"] = baseline_ppl
RESULTS["Zero-shot FLAN-T5-small"] = summ
PER_EXAMPLE["Zero-shot FLAN-T5-small"] = per

display(pd.DataFrame({"query": test_eval_df["instruction"].head(4), "zero-shot answer": baseline_preds[:4],
                      "reference": test_eval_df["target"].head(4)}))
display(pd.Series(summ, name="Zero-shot baseline").round(4).to_frame())

del base_model; gc.collect(); torch.cuda.empty_cache()

## 6. Hyper-parameter Tuning of the LLM (Optuna)
We search the **training** hyper-parameters that matter most when fine-tuning a T5 model:

| Hyper-parameter | Search range | Why it matters |
|---|---|---|
| Learning rate | 5e-5 → 2e-3 (log scale) | T5 usually likes much higher rates than BERT-style models; too high diverges, too low underfits |
| Batch size | 8 / 16 / 32 | Trade-off between gradient noise, speed and memory |
| Weight decay | 0 → 0.1 | Regularisation |
| Warm-up ratio | 0 → 0.1 | Stabilises early training |
| LR scheduler | linear / cosine | Shape of the decay |

Each trial fine-tunes a fresh model on a **subset** and is scored by **validation loss** (lower is better).
The first trial is deliberately the *default Hugging Face-style setting* so we can quantify the gain from tuning.

In [ ]:
def build_trainer(model_, train_ds, eval_ds, hp, epochs, out_dir, final=False):
    steps_per_epoch = math.ceil(len(train_ds) / hp["batch_size"])
    args = Seq2SeqTrainingArguments(
        output_dir=out_dir, num_train_epochs=epochs,
        learning_rate=hp["learning_rate"], weight_decay=hp["weight_decay"],
        per_device_train_batch_size=hp["batch_size"], per_device_eval_batch_size=64,
        warmup_steps=int(hp["warmup_ratio"] * steps_per_epoch * epochs),
        lr_scheduler_type=hp["lr_scheduler_type"],
        eval_strategy="epoch", save_strategy="epoch" if final else "no", save_total_limit=1,
        load_best_model_at_end=final, metric_for_best_model="eval_loss", greater_is_better=False,
        logging_steps=max(10, steps_per_epoch // 10), predict_with_generate=False,
        bf16=USE_BF16, report_to="none", seed=SEED, disable_tqdm=not final)
    kwargs = dict(model=model_, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
                  data_collator=DataCollatorForSeq2Seq(tokenizer, model=model_, pad_to_multiple_of=8),
                  callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] if final else None)
    try:
        return Seq2SeqTrainer(processing_class=tokenizer, **kwargs)
    except TypeError:                       # older transformers versions
        return Seq2SeqTrainer(tokenizer=tokenizer, **kwargs)

tune_train = train_tok.shuffle(seed=SEED).select(range(min(CFG["tune_train_n"], len(train_tok))))
tune_val = val_tok.select(range(min(CFG["tune_val_n"], len(val_tok))))
print(f"Tuning on {len(tune_train):,} training and {len(tune_val):,} validation examples, {CFG['n_trials']} trials.")

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
trial_times = {}

def objective(trial):
    hp = dict(
        learning_rate=trial.suggest_float("learning_rate", 5e-5, 2e-3, log=True),
        batch_size=trial.suggest_categorical("batch_size", [8, 16, 32]),
        weight_decay=trial.suggest_float("weight_decay", 0.0, 0.1),
        warmup_ratio=trial.suggest_float("warmup_ratio", 0.0, 0.1),
        lr_scheduler_type=trial.suggest_categorical("lr_scheduler_type", ["linear", "cosine"]),
    )
    set_seed(SEED)
    m = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL).to(device)
    trainer = build_trainer(m, tune_train, tune_val, hp, CFG["tune_epochs"], os.path.join(OUTPUT_DIR, "tune_tmp"))
    t0 = time.time()
    try:
        trainer.train()
        loss = trainer.evaluate()["eval_loss"]
    except torch.cuda.OutOfMemoryError:
        raise optuna.TrialPruned()
    finally:
        del trainer, m; gc.collect(); torch.cuda.empty_cache()
    trial_times[trial.number] = time.time() - t0
    return loss

def progress_cb(study_, trial):
    print(f"Trial {trial.number:>2} | val loss = {trial.value:.4f} | ppl = {math.exp(trial.value):6.2f} | "
          f"{trial_times.get(trial.number, 0):5.0f}s | {trial.params}")

study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED, n_startup_trials=4))
# Trial 0 = default-style baseline so the benefit of tuning is measurable
study.enqueue_trial({"learning_rate": 5e-5, "batch_size": 8, "weight_decay": 0.0, "warmup_ratio": 0.0, "lr_scheduler_type": "linear"})
study.optimize(objective, n_trials=CFG["n_trials"], callbacks=[progress_cb])

In [ ]:
trials_df = study.trials_dataframe()
trials_df = trials_df[trials_df["state"] == "COMPLETE"].copy()
trials_df.columns = [c.replace("params_", "") for c in trials_df.columns]
trials_df = trials_df.rename(columns={"value": "val_loss", "number": "trial"})
trials_df["perplexity"] = np.exp(trials_df["val_loss"])
trials_df["label"] = np.where(trials_df["trial"] == 0, "default-style", "tuned search")
cols = ["trial", "label", "learning_rate", "batch_size", "weight_decay", "warmup_ratio", "lr_scheduler_type", "val_loss", "perplexity"]
display(trials_df[cols].sort_values("val_loss").round(5).reset_index(drop=True))

BEST_HP = dict(study.best_params)
print("\n✅ BEST HYPER-PARAMETERS:", BEST_HP)
default_loss = trials_df.loc[trials_df["trial"] == 0, "val_loss"].iloc[0]
print(f"Default-style loss {default_loss:.4f} (ppl {math.exp(default_loss):.2f})  →  tuned loss {study.best_value:.4f} (ppl {math.exp(study.best_value):.2f})  "
      f"= {(1 - math.exp(study.best_value) / math.exp(default_loss)) * 100:.1f}% lower perplexity")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5))

# (1) optimisation history
ax = axes[0]
ax.bar(trials_df["trial"], trials_df["val_loss"], color=np.where(trials_df["trial"] == 0, "#e76f51", "#8ecae6"))
ax.plot(trials_df["trial"], trials_df["val_loss"].cummin(), "k-o", label="best so far")
ax.set_xlabel("trial"); ax.set_ylabel("validation loss"); ax.set_title("Optimisation history (orange = default-style)"); ax.legend()

# (2) learning rate vs loss
ax = axes[1]
sc = ax.scatter(trials_df["learning_rate"], trials_df["val_loss"], c=trials_df["batch_size"], cmap="viridis", s=120, edgecolor="k")
ax.set_xscale("log"); ax.set_xlabel("learning rate (log)"); ax.set_ylabel("validation loss"); ax.set_title("Learning rate vs loss (colour = batch size)")
plt.colorbar(sc, ax=ax, label="batch size")

# (3) hyper-parameter importance
ax = axes[2]
try:
    imp = optuna.importance.get_param_importances(study)
    sns.barplot(x=list(imp.values()), y=list(imp.keys()), hue=list(imp.keys()), palette="rocket", legend=False, ax=ax)
    ax.set_title("Hyper-parameter importance"); ax.set_xlabel("relative importance")
except Exception as e:
    ax.text(0.5, 0.5, f"Importance unavailable\n({type(e).__name__})", ha="center"); ax.axis("off")
plt.tight_layout(); plt.show()

try:
    optuna.visualization.plot_parallel_coordinate(study, target_name="validation loss").show()
except Exception as e:
    print("Parallel-coordinate plot skipped:", e)

**How to read this:** the orange bar is the default-style configuration. Any bar lower than it is an improvement from tuning.
The scatter shows the learning-rate sweet-spot; the importance chart shows which knob mattered most (with few trials treat it as indicative, not definitive).

## 7. Final Fine-Tuning with the Best Hyper-parameters
Now we train a fresh model with `BEST_HP` on the training split (profile-dependent size), validate every epoch,
keep the **best checkpoint** and stop early if validation loss stops improving.

In [ ]:
final_train = train_tok if CFG["final_train_n"] is None else train_tok.shuffle(seed=SEED).select(range(min(CFG["final_train_n"], len(train_tok))))
print(f"Final training on {len(final_train):,} examples for up to {CFG['final_epochs']} epochs with {BEST_HP}")

set_seed(SEED)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL).to(device)
trainer = build_trainer(model, final_train, val_tok, BEST_HP, CFG["final_epochs"], os.path.join(OUTPUT_DIR, "checkpoints"), final=True)

t0 = time.time()
train_result = trainer.train()
train_minutes = (time.time() - t0) / 60
model = trainer.model
print(f"\nTraining finished in {train_minutes:.1f} min | train samples/sec: {train_result.metrics.get('train_samples_per_second', float('nan')):.1f}")

In [ ]:
hist = trainer.state.log_history
tr = pd.DataFrame([(h["step"], h["loss"]) for h in hist if "loss" in h], columns=["step", "loss"])
ev = pd.DataFrame([(h["epoch"], h["eval_loss"]) for h in hist if "eval_loss" in h], columns=["epoch", "eval_loss"])

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
axes[0].plot(tr["step"], tr["loss"], alpha=0.35, label="training loss (raw)")
axes[0].plot(tr["step"], tr["loss"].rolling(5, min_periods=1).mean(), lw=2.5, label="training loss (smoothed)")
axes[0].set_xlabel("step"); axes[0].set_ylabel("loss"); axes[0].set_title("Training loss curve"); axes[0].legend()

axes[1].plot(ev["epoch"], ev["eval_loss"], "o-", lw=2.5, color="#e76f51", label="validation loss")
for _, r in ev.iterrows():
    axes[1].annotate(f"ppl {math.exp(r['eval_loss']):.2f}", (r["epoch"], r["eval_loss"]), textcoords="offset points", xytext=(0, 9), ha="center")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("loss"); axes[1].set_title("Validation loss per epoch (with perplexity)"); axes[1].legend()
plt.tight_layout(); plt.show()

val_loss = compute_eval_loss(model, val_tok); test_loss = compute_eval_loss(model, test_tok)
print(f"Validation loss {val_loss:.4f} (ppl {math.exp(val_loss):.2f}) | Test loss {test_loss:.4f} (ppl {math.exp(test_loss):.2f})")
print(f"Gap between last training loss and validation loss: {abs(tr['loss'].iloc[-1] - val_loss):.3f} (small gap = no strong over-fitting)")

SAVE_DIR = os.path.join(OUTPUT_DIR, "final_model")
model.save_pretrained(SAVE_DIR); tokenizer.save_pretrained(SAVE_DIR)
print("Model saved to", SAVE_DIR)

## 8. Decoding (Generation) Hyper-parameter Tuning
Fine-tuning fixes the *weights*; **how text is generated** matters just as much. The original model **looped** with greedy decoding.
We search beam count, n-gram blocking and repetition penalty, and also test two sampling settings, scoring each on the **validation** set
(never the test set — that would be tuning on the exam).

**Selection rule:** among configurations whose repetition rate is ≤ 15%, pick the highest ROUGE-L.

In [ ]:
decode_val = val_df.sample(n=min(CFG["decode_val_n"], len(val_df)), random_state=SEED).reset_index(drop=True)

decode_configs = {}
for beams in [1, 3, 5]:
    for nr in [0, 3]:
        for rp in [1.0, 1.3]:
            decode_configs[f"beams={beams} | no_repeat={nr} | rep_pen={rp}"] = dict(num_beams=beams, no_repeat_ngram_size=nr, repetition_penalty=rp, do_sample=False)
decode_configs["sampling T=0.7 top_p=0.9"] = dict(do_sample=True, temperature=0.7, top_p=0.9, num_beams=1, no_repeat_ngram_size=3)
decode_configs["sampling T=1.0 top_p=0.95"] = dict(do_sample=True, temperature=1.0, top_p=0.95, num_beams=1, no_repeat_ngram_size=3)

rows = []
for name, cfg in tqdm(decode_configs.items(), desc="decoding configs"):
    set_seed(SEED)
    t0 = time.time()
    preds = generate_batch(model, list(decode_val["source"]), **cfg)
    lat = (time.time() - t0) / len(preds)
    s, _ = compute_metrics_suite(preds, decode_val["target"], use_bert=False)
    rows.append({"config": name, "ROUGE-L": s["ROUGE-L"], "ROUGE-1": s["ROUGE-1"], "BLEU": s["BLEU"], "METEOR": s["METEOR"],
                 "Repetition rate": s["Repetition rate"], "Looping rate": s["Looping rate"], "Length ratio": s["Length ratio"],
                 "sec/answer": lat})
decode_df = pd.DataFrame(rows).sort_values("ROUGE-L", ascending=False).reset_index(drop=True)
display(decode_df.round(4))

In [ ]:
REP_CAP = 0.15
eligible = decode_df[decode_df["Repetition rate"] <= REP_CAP]
best_row = (eligible if len(eligible) else decode_df.sort_values("Repetition rate")).iloc[0]
BEST_GEN = dict(decode_configs[best_row["config"]])
print("✅ BEST DECODING CONFIG:", best_row["config"], "->", BEST_GEN)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
d = decode_df.sort_values("ROUGE-L")
colors = ["#2a9d8f" if c == best_row["config"] else "#a8dadc" for c in d["config"]]
axes[0].barh(d["config"], d["ROUGE-L"], color=colors); axes[0].set_xlabel("ROUGE-L (validation)"); axes[0].set_title("Decoding configs ranked by ROUGE-L (green = selected)")
sc = axes[1].scatter(decode_df["Repetition rate"], decode_df["ROUGE-L"], c=decode_df["sec/answer"], cmap="plasma", s=140, edgecolor="k")
axes[1].axvline(REP_CAP, color="red", ls="--", label=f"repetition cap = {REP_CAP}")
axes[1].set_xlabel("Repetition rate (lower is better)"); axes[1].set_ylabel("ROUGE-L"); axes[1].set_title("Quality vs repetition (colour = seconds/answer)")
plt.colorbar(sc, ax=axes[1], label="sec / answer"); axes[1].legend()
plt.tight_layout(); plt.show()

## 9. Final Evaluation on the Held-out Test Set
Three systems are compared on the **same** unseen test questions:
1. Zero-shot FLAN-T5-small (no fine-tuning)
2. Fine-tuned model + greedy decoding (what the original submission effectively did)
3. Fine-tuned model + **tuned decoding**

In [ ]:
test_sources = list(test_eval_df["source"]); test_refs = test_eval_df["target"]

ft_greedy = generate_batch(model, test_sources, **GREEDY)
set_seed(SEED)
ft_tuned = generate_batch(model, test_sources, **BEST_GEN)

ft_ppl = math.exp(test_loss)
for name, preds in [("Fine-tuned + greedy", ft_greedy), ("Fine-tuned + tuned decoding", ft_tuned)]:
    s, per = compute_metrics_suite(preds, test_refs)
    s["Perplexity"] = ft_ppl
    RESULTS[name] = s; PER_EXAMPLE[name] = per

results_df = pd.DataFrame(RESULTS).T
display(results_df.round(4))
results_df.round(5).to_csv(os.path.join(OUTPUT_DIR, "evaluation_results.csv"))

In [ ]:
quality_cols = ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BLEU", "METEOR", "BERTScore-F1"]
diag_cols = ["Repetition rate", "Looping rate", "Distinct-2", "Length ratio"]
palette = ["#b0b0b0", "#f4a261", "#2a9d8f"]

fig, axes = plt.subplots(1, 3, figsize=(21, 5.5), gridspec_kw={"width_ratios": [3, 2.2, 1]})
results_df[quality_cols].T.plot(kind="bar", ax=axes[0], color=palette, width=0.8)
axes[0].set_title("Answer quality (higher is better)"); axes[0].set_ylabel("score"); axes[0].tick_params(axis="x", rotation=25)
results_df[diag_cols].T.plot(kind="bar", ax=axes[1], color=palette, width=0.8, legend=False)
axes[1].axhline(1.0, color="grey", ls=":", lw=1)
axes[1].set_title("Generation diagnostics (repetition/looping: lower is better)"); axes[1].tick_params(axis="x", rotation=25)
ppl = results_df["Perplexity"]
axes[2].bar(range(len(ppl)), ppl.values, color=palette); axes[2].set_xticks(range(len(ppl))); axes[2].set_xticklabels(["zero-shot", "FT greedy", "FT tuned"], rotation=20)
for i, v in enumerate(ppl.values): axes[2].text(i, v, f"{v:.1f}", ha="center", va="bottom")
axes[2].set_title("Perplexity (lower is better)")
plt.tight_layout(); plt.show()

base, best = results_df.iloc[0], results_df.iloc[-1]
print("Improvement of the final system over the zero-shot baseline:")
for m in ["ROUGE-L", "BLEU", "METEOR", "BERTScore-F1"]:
    print(f"  {m:13s}: {base[m]:.3f} → {best[m]:.3f}  ({(best[m] / max(base[m], 1e-9) - 1) * 100:+.1f}%)")
print(f"  Perplexity   : {base['Perplexity']:.2f} → {best['Perplexity']:.2f}")
print(f"  Looping rate : {results_df.loc['Fine-tuned + greedy', 'Looping rate']:.1%} (greedy) → {best['Looping rate']:.1%} (tuned decoding)")

### How to interpret the metrics (business view)
- **Perplexity ↓** — the model finds real bank answers *less surprising*; a general-purpose model does not know bank phrasing.
- **ROUGE / METEOR ↑** — how much of the reference content and wording the bot reproduces. For customer support, higher overlap means answers stay on-script and compliant.
- **BLEU ↑** — precision of exact phrases; low-ish BLEU is normal for open-ended answers with many valid phrasings.
- **BERTScore ↑** — meaning-level similarity; the best single indicator that the bot "says the right thing" even in different words.
- **Repetition / looping ↓ and Length ratio ≈ 1** — protect the customer experience: no rambling, no cut-off or bloated answers.
- **Placeholder recall ↑** — the bot keeps fields such as `{{Account Number}}` that the real banking system fills in.

> Reference-based metrics are an *approximation* of quality — pair them with the qualitative review below before drawing conclusions.

## 10. Error Analysis & Robustness
### 10.1 Score distributions

In [ ]:
analysis = test_eval_df[["flags"] if has_flags else []].copy() if has_flags else pd.DataFrame(index=test_eval_df.index)
analysis["instruction"] = test_eval_df["instruction"]; analysis["category"] = test_eval_df["category"]; analysis["intent"] = test_eval_df["intent"]
analysis["reference"] = test_eval_df["target"]; analysis["prediction"] = ft_tuned
analysis = analysis.join(PER_EXAMPLE["Fine-tuned + tuned decoding"])

fig, axes = plt.subplots(1, 3, figsize=(20, 4.8))
for name, col in [("Zero-shot FLAN-T5-small", "#b0b0b0"), ("Fine-tuned + greedy", "#f4a261"), ("Fine-tuned + tuned decoding", "#2a9d8f")]:
    sns.kdeplot(PER_EXAMPLE[name]["rougeL"], fill=True, alpha=0.35, color=col, label=name, ax=axes[0])
axes[0].set_title("Per-example ROUGE-L distribution"); axes[0].set_xlabel("ROUGE-L"); axes[0].legend()

sns.histplot(analysis["bertscore_f1"], bins=30, kde=True, color="#2a9d8f", ax=axes[1])
axes[1].set_title("Per-example BERTScore-F1 (final system)"); axes[1].set_xlabel("BERTScore-F1")

mx = max(analysis["ref_words"].max(), analysis["pred_words"].max())
axes[2].scatter(analysis["ref_words"], analysis["pred_words"], alpha=0.35, s=18, color="#264653")
axes[2].plot([0, mx], [0, mx], "r--", label="ideal (same length)")
axes[2].set_xlabel("reference words"); axes[2].set_ylabel("generated words"); axes[2].set_title("Generated vs reference length"); axes[2].legend()
plt.tight_layout(); plt.show()

### 10.2 Performance by category and by intent

In [ ]:
by_cat = analysis.groupby("category")[["rougeL", "bertscore_f1"]].mean().sort_values("rougeL")
fig, axes = plt.subplots(1, 2, figsize=(18, max(4.5, 0.45 * len(by_cat) + 2)))
by_cat.plot(kind="barh", ax=axes[0], color=["#e9c46a", "#2a9d8f"], width=0.8)
axes[0].set_title("Mean ROUGE-L & BERTScore by category"); axes[0].set_ylabel("")

by_int = analysis.groupby("intent").agg(rougeL=("rougeL", "mean"), n=("rougeL", "size"))
by_int = by_int[by_int["n"] >= 5].sort_values("rougeL")
if len(by_int) >= 2:
    show = pd.concat([by_int.head(8), by_int.tail(8)]) if len(by_int) > 16 else by_int
    sns.barplot(x=show["rougeL"], y=show.index, hue=show.index, palette="RdYlGn", legend=False, ax=axes[1])
    axes[1].set_title("Weakest and strongest intents (ROUGE-L, ≥5 test samples)"); axes[1].set_ylabel("")
else:
    axes[1].axis("off")
plt.tight_layout(); plt.show()

### 10.3 Robustness to noisy / varied customer phrasing

In [ ]:
if has_flags and "flags" in analysis.columns:
    rows = []
    for letter in sorted(set("".join(analysis["flags"].dropna().astype(str)))):
        mask = analysis["flags"].astype(str).str.contains(letter, regex=False)
        if mask.sum() >= 15:
            rows.append({"flag": letter, "n": int(mask.sum()), "ROUGE-L": analysis.loc[mask, "rougeL"].mean(), "BERTScore-F1": analysis.loc[mask, "bertscore_f1"].mean()})
    flag_df = pd.DataFrame(rows)
    if len(flag_df):
        flag_df.set_index("flag")[["ROUGE-L", "BERTScore-F1"]].plot(kind="bar", figsize=(10, 4.5), color=["#e9c46a", "#2a9d8f"])
        plt.title("Performance per linguistic-variation flag (does noisy phrasing hurt?)"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()
        display(flag_df.round(4))
    else:
        print("Not enough samples per flag letter in this test subset (use PROFILE='full').")
else:
    print("This dataset has no 'flags' column - robustness-by-flag analysis skipped.")

### 10.4 Qualitative review — worst cases and random samples

In [ ]:
show_cols = ["instruction", "prediction", "reference", "rougeL", "bertscore_f1"]
print("🔻 10 WORST answers by ROUGE-L:")
display(analysis.sort_values("rougeL").head(10)[show_cols].round(3))
print("\n🎲 8 RANDOM answers:")
display(analysis.sample(8, random_state=SEED)[show_cols].round(3))
print(f"\nLooping answers (>30% repeated trigrams): {(analysis['repetition'] > 0.3).sum()} of {len(analysis)}")

### 10.5 Inference latency (production readiness)

In [ ]:
queries = list(test_eval_df["source"].head(20)); lat = []
model.eval()
for qy in queries:
    enc = tokenizer(qy, return_tensors="pt", truncation=True, max_length=MAX_SRC).to(device)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=MAX_TGT, **BEST_GEN)
    lat.append(time.time() - t0)
lat = np.array(lat)
print(f"Single-query latency on {device}: median {np.median(lat)*1000:.0f} ms | p95 {np.percentile(lat, 95)*1000:.0f} ms | mean {lat.mean()*1000:.0f} ms")
print(f"Model size: {sum(p.numel() for p in model.parameters())/1e6:.1f} M parameters")

## 11. Interactive Gradio Application (use this in the demo video)
The app has three tabs:
- **💬 Chat** — ask banking questions; open *Generation settings* to change beams, repetition penalty, temperature live.
- **🧪 Evaluate a response** — type a question and a reference answer; the app generates a reply and computes ROUGE/BLEU/METEOR/repetition **live**.
- **📊 Model report** — the final comparison table (zero-shot vs fine-tuned vs tuned decoding).

> The model answers each question independently (single-turn); chat history is shown for convenience only. Do not enter real account details.

In [ ]:
import gradio as gr
print("Gradio version:", gr.__version__)

@torch.no_grad()
def answer(question, num_beams=None, no_repeat=None, rep_pen=None, temperature=0.0, max_tokens=None):
    kw = dict(BEST_GEN)
    if num_beams is not None:  kw["num_beams"] = int(num_beams)
    if no_repeat is not None:  kw["no_repeat_ngram_size"] = int(no_repeat)
    if rep_pen is not None:    kw["repetition_penalty"] = float(rep_pen)
    if temperature and temperature > 0:
        kw.update(do_sample=True, temperature=float(temperature), top_p=0.95, num_beams=1)
    else:
        kw.update(do_sample=False); kw.pop("temperature", None); kw.pop("top_p", None)
    enc = tokenizer(PREFIX + clean_text(question), return_tensors="pt", truncation=True, max_length=MAX_SRC).to(device)
    model.eval(); t0 = time.time()
    out = model.generate(**enc, max_new_tokens=int(max_tokens or MAX_TGT), **kw)
    return tokenizer.decode(out[0], skip_special_tokens=True), time.time() - t0

def respond(message, history, nb, nr, rp, temp, mt):
    history = history or []
    if not message or not message.strip():
        return history, "", ""
    ans, dt = answer(message, nb, nr, rp, temp, mt)
    history = history + [{"role": "user", "content": message}, {"role": "assistant", "content": ans}]
    return history, "", f"⏱️ {dt * 1000:.0f} ms"

def evaluate_live(question, reference):
    if not question.strip() or not reference.strip():
        return "Please enter both a question and a reference answer.", {}
    pred, dt = answer(question)
    sc = _rouge.score(reference, pred)
    try:    met = meteor_score([reference.split()], pred.split())
    except Exception: met = float("nan")
    metrics = {"ROUGE-1": round(sc["rouge1"].fmeasure, 4), "ROUGE-2": round(sc["rouge2"].fmeasure, 4), "ROUGE-L": round(sc["rougeL"].fmeasure, 4),
               "BLEU": round(sacrebleu.sentence_bleu(pred, [reference]).score / 100, 4), "METEOR": round(float(met), 4),
               "Repetition rate": round(trigram_repetition(pred), 4),
               "Length ratio": round(len(pred.split()) / max(1, len(reference.split())), 3), "Latency (ms)": round(dt * 1000)}
    return pred, metrics

def make_chatbot():
    try:    return gr.Chatbot(type="messages", height=430, label="Banking Assistant")
    except TypeError: return gr.Chatbot(height=430, label="Banking Assistant")

EXAMPLES = ["How can I activate my debit card?", "I lost my credit card, what should I do?", "How do I check my account balance?",
            "where is the nearest atm", "I want to apply for a personal loan", "how do i dispute a charge on my statement",
            "What documents do I need to open a savings account?"]

with gr.Blocks(title="Retail Banking Assistant", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🏦 Retail Banking Assistant\nFine-tuned **FLAN-T5** on the Bitext retail-banking dataset · tuned hyper-parameters · tuned decoding")
    with gr.Tabs():
        with gr.Tab("💬 Chat"):
            chatbot = make_chatbot()
            msg = gr.Textbox(label="Your banking question", placeholder="e.g. How can I activate my debit card?")
            with gr.Row():
                send = gr.Button("Send", variant="primary"); clear = gr.ClearButton([msg, chatbot], value="Clear")
            latency = gr.Markdown()
            with gr.Accordion("⚙️ Generation settings", open=False):
                nb = gr.Slider(1, 8, value=BEST_GEN.get("num_beams", 1), step=1, label="Beam search width")
                nr = gr.Slider(0, 5, value=BEST_GEN.get("no_repeat_ngram_size", 0), step=1, label="No-repeat n-gram size (0 = off)")
                rp = gr.Slider(1.0, 2.0, value=float(BEST_GEN.get("repetition_penalty", 1.0)), step=0.05, label="Repetition penalty")
                temp = gr.Slider(0.0, 1.5, value=0.0, step=0.05, label="Temperature (0 = deterministic)")
                mt = gr.Slider(32, MAX_TGT, value=min(256, MAX_TGT), step=16, label="Max new tokens")
            gr.Examples(examples=EXAMPLES, inputs=msg, label="Try these")
            inputs = [msg, chatbot, nb, nr, rp, temp, mt]
            msg.submit(respond, inputs, [chatbot, msg, latency]); send.click(respond, inputs, [chatbot, msg, latency])
        with gr.Tab("🧪 Evaluate a response"):
            gr.Markdown("Enter a customer question **and** the ideal answer. The model replies and the LLM-evaluation metrics are computed live.")
            q_in = gr.Textbox(label="Customer question"); ref_in = gr.Textbox(label="Reference (ideal) answer", lines=4)
            run_btn = gr.Button("Generate & evaluate", variant="primary")
            pred_out = gr.Textbox(label="Model answer", lines=6); met_out = gr.JSON(label="Metrics")
            run_btn.click(evaluate_live, [q_in, ref_in], [pred_out, met_out])
        with gr.Tab("📊 Model report"):
            gr.Markdown("Test-set comparison (held-out questions the model never saw in training).")
            gr.Dataframe(value=results_df.round(4).reset_index().rename(columns={"index": "model"}), interactive=False)
            gr.Markdown(f"**Best training hyper-parameters:** `{BEST_HP}`  \n**Best decoding config:** `{BEST_GEN}`")

demo.queue().launch(share=True, debug=False)   # non-blocking: the app stays live while the remaining cells run
print("👆 Open the public link above for the live prototype (valid ~1 week while this runtime stays connected).")

## 12. Save Artifacts & Final Summary

In [ ]:
artifacts = {"base_model": BASE_MODEL, "profile": PROFILE, "best_training_hyperparameters": BEST_HP, "best_decoding_config": BEST_GEN,
             "max_source_tokens": MAX_SRC, "max_target_tokens": MAX_TGT, "train_minutes": round(train_minutes, 2),
             "train_rows": len(train_df), "val_rows": len(val_df), "test_rows": len(test_df)}
with open(os.path.join(OUTPUT_DIR, "run_summary.json"), "w") as f:
    json.dump(artifacts, f, indent=2, default=str)
decode_df.round(5).to_csv(os.path.join(OUTPUT_DIR, "decoding_search.csv"), index=False)
import shutil   # copy run settings + results into the model folder so the standalone app (app.py) is self-contained
for fn in ["run_summary.json", "evaluation_results.csv"]:
    shutil.copy(os.path.join(OUTPUT_DIR, fn), os.path.join(SAVE_DIR, fn))
trials_df.to_csv(os.path.join(OUTPUT_DIR, "optuna_trials.csv"), index=False)

best = results_df.loc["Fine-tuned + tuned decoding"]; base = results_df.loc["Zero-shot FLAN-T5-small"]
display(Markdown(f"""
### ✅ Final results (held-out test set, {len(test_eval_df):,} questions)
| Metric | Zero-shot | **Fine-tuned + tuned decoding** |
|---|---|---|
| Perplexity ↓ | {base['Perplexity']:.2f} | **{best['Perplexity']:.2f}** |
| ROUGE-L ↑ | {base['ROUGE-L']:.3f} | **{best['ROUGE-L']:.3f}** |
| BLEU ↑ | {base['BLEU']:.3f} | **{best['BLEU']:.3f}** |
| METEOR ↑ | {base['METEOR']:.3f} | **{best['METEOR']:.3f}** |
| BERTScore-F1 ↑ | {base['BERTScore-F1']:.3f} | **{best['BERTScore-F1']:.3f}** |
| Looping rate ↓ | {base['Looping rate']:.1%} | **{best['Looping rate']:.1%}** |

**Best training hyper-parameters:** `{BEST_HP}`  
**Best decoding configuration:** `{BEST_GEN}`  
All files saved to `{OUTPUT_DIR}`.
"""))

In [ ]:
# 💾 Save THIS notebook together with all charts/tables/outputs to Drive (this is the file to submit).
# Run it after every cell above has finished.
try:
    from google.colab import _message
    nb_json = _message.blocking_request("get_ipynb", request="", timeout_sec=600)["ipynb"]
    out_path = os.path.join(OUTPUT_DIR, "Banking_LLM_Chatbot_WITH_OUTPUTS.ipynb")
    with open(out_path, "w") as f:
        json.dump(nb_json, f)
    print("✅ Saved notebook with outputs to:", out_path)
    print("   (Also available via File → Download → Download .ipynb)")
except Exception as e:
    print("Automatic save not available here:", e)
    print("➡️  Use File → Download → Download .ipynb — Colab includes all outputs in the download.")

## 13. Conclusion, Limitations & Future Work
**What was achieved:** a domain-specific retail-banking assistant built by (1) auditing and exploring the data, (2) cleaning it with a leakage-safe, documented pipeline,
(3) tuning both training and decoding hyper-parameters, (4) evaluating with overlap, semantic and generation-quality metrics against a zero-shot baseline, and (5) serving it through an interactive Gradio app.

**Limitations (be upfront about these in your video):**
- FLAN-T5-*small* (~77 M parameters) has limited capacity; answers are fluent but can be generic. `flan-t5-base` or LoRA-tuned larger models would likely score higher.
- The Bitext data is **synthetic/templated**, so scores overstate performance on truly open-ended real customer chats.
- Reference-based metrics under-credit valid paraphrases; a human or LLM-as-judge evaluation would strengthen the study.
- The bot is single-turn and has no grounding in a live knowledge base — it must not be used for real financial advice.
- The tuning search is small (`quick` profile); `full` gives a more reliable optimum.

**Future work:** parameter-efficient fine-tuning (LoRA), retrieval-augmented generation over bank FAQs, multi-turn dialogue, intent classification + guard-rails, and human evaluation.

---
### 🎬 Demo-video checklist (covers the feedback)
1. Show the **EDA** charts and the key-findings printout (Section 2) — explain the truncation/imbalance insights.
2. Show the **preprocessing funnel** and split/leakage check (Section 3).
3. Show the **Optuna table + plots** and the *default vs tuned* improvement (Section 6) and the **decoding search** (Section 8).
4. Show the **metrics comparison chart** and explain each metric (Section 9), then the error analysis (Section 10).
5. **Live Gradio demo:** ask 3–4 questions in *Chat*, change a setting (e.g. repetition penalty), then use *Evaluate a response* to show live ROUGE/BLEU/METEOR, and finish on the *Model report* tab.